# Heston calibration — chaos with a Legendre basis

Run one reproducible calibration, evaluate the fit on calibration and held-out maturities. The chaos model uses the general-basis conditional-expectation implementation and first- and second-moment control variates.

Use **Restart Kernel → Run All** for a fresh fit. Set `CONFIG["initialization"]["type"]` to `"random"` to compare against the Heston warm start. Existing saved coefficients remain untouched until the final save cell runs.


## 1. Imports and data

Paths work from this notebook's directory or the repository root. The pricing module is reloaded when this cell runs so an old class definition is not reused.


In [ ]:
from pathlib import Path
import hashlib
import importlib
import json
import pickle
import random
import sys
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
from IPython.display import Markdown, display

ROOT = next(
    (path for path in (Path.cwd(), *Path.cwd().parents)
     if (path / "wiener_chaos/core/model.py").exists()),
    None,
)
if ROOT is None:
    raise FileNotFoundError("Open this notebook from within the project directory.")
STUDY_DIR = ROOT / "Heston" / "Legendre"
if str(ROOT) in sys.path:
    sys.path.remove(str(ROOT))
sys.path.insert(0, str(ROOT))

from wiener_chaos.experiments.calibration import save_artifacts
from wiener_chaos.experiments.provenance import source_fingerprint
from wiener_chaos.pricing.options import compute_ivs
from wiener_chaos.pricing import heston as heston_module
heston_module = importlib.reload(heston_module)
from wiener_chaos.core import model as pricing_module
pricing_module = importlib.reload(pricing_module)
ModelStock = pricing_module.ModelStock
print(f"Pricing module loaded: {pricing_module.__file__}")

DATA_PATH = ROOT / "Heston" / "heston_data.pkl"
with DATA_PATH.open("rb") as handle:
    data = pickle.load(handle)

pd.set_option("display.max_columns", 12)
plt.rcParams.update({"figure.figsize": (10, 4), "axes.grid": True, "grid.alpha": 0.25})


## 2. Calibration settings

Use `P=2`, `d=2`, and 10 Legendre functions per driver. Heston initialization
starts from the surface-derived guess and uses 20,000 × 4 projection paths,
101 simulation time points, and a maximum of 100 optimizer iterations.
Calibration and test maturities retain their own Monte Carlo allocations.

Chaos calibration uses `lr=1e-4`, up to 10,000 steps, training-loss monitoring,
and a `1e-6` early-stopping loss gate. Degree-2 moment control variates are active
immediately. Both regularization weights are zero, including `anchor_lambda=0`.
GPU selection is controlled by `CUDA_DEVICE_INDEX` below.

### Optional heuristic-only Heston initialization

Leave `CONFIG["initialization"]["fit_heston"] = True` for the current fitted-Heston
warm start. Set it to `False` to use the existing surface-based Heston heuristic
directly, skipping Heston optimization. The chaos projection and subsequent chaos
calibration still run with the same budgets. `heston_fit` is empty when the Heston
optimizer is skipped. Use the same choice in calibration and calibration study.


In [ ]:
CUDA_DEVICE_INDEX = 0  # Index of the GPU used for all model computations.
if torch.cuda.is_available() and CUDA_DEVICE_INDEX >= torch.cuda.device_count():
    raise RuntimeError(f"CUDA device {CUDA_DEVICE_INDEX} is unavailable.")
DEVICE = f"cuda:{CUDA_DEVICE_INDEX}" if torch.cuda.is_available() else "cpu"
if DEVICE == "cpu":
    # Large thread pools can slow the many small matrix operations in this study.
    torch.set_num_threads(1)

CONFIG = {
    # Master seed for reproducible initialization and sampling.
    "base_seed": 42,
    # Device used for model tensors and pricing.
    "device": DEVICE,
    # Time-basis family for the Brownian chaos expansion.
    "basis": "legendre",
    # Number of Brownian drivers.
    "d": 2,
    # Maximum total chaos order.
    "P": 2,
    # Number of time-basis functions per driver.
    "M": 10,
    # Diagonal jitter that stabilizes basis covariance factorizations.
    "base_jitter": 1e-12,
    # How the starting chaos coefficients are obtained.
    "initialization": {
        # Choose a fitted-Heston projection or a random chaos start.
        "type": "calibrate_heston",
        # Starting point for the Heston parameter optimizer.
        "initial_guess": "surface",
        # False uses the surface heuristic directly, without Heston optimization.
        "fit_heston": True,
        # Use the same plain projection for both chaos bases.
        "projection_control": "none",
        "projection_antithetic": False,
        "projection_repetitions": 1,
        # Brownian paths in each terminal-factor projection batch.
        "N": 20_000,
        # Number of projection batches; total paths are N times Ns.
        "Ns": 4,
        # Number of time-grid points in each Heston simulation.
        "n_steps_heston": 101,
        # Settings passed to the Heston optimizer.
        "options": {
            # Maximum Heston optimizer iterations.
            "maxiter": 100,
        },
    },
    # Pricing allocations at calibration maturities.
    "pricing_calibr": {
        # Integration method for each calibration maturity (MC or Gauss-Hermite).
        "method": ["MC"] * 7,
        # MC paths per calibration maturity; None for quadrature.
        "N": [1_000_000, 500_000] + [100_000] * 5,
        # Independent MC batches per calibration maturity.
        "Ns": [1] * 7,
        # Gauss-Hermite nodes per dimension; None for MC.
        "nodes": [None] * 7,
    },
    # Pricing allocations at held-out test maturities.
    "pricing_test": {
        # Integration method for each held-out maturity.
        "method": ["MC"] * 6,
        # MC paths per held-out maturity; None for quadrature.
        "N": [1_000_000, 1_000_000] + [100_000] * 4,
        # Independent MC batches per held-out maturity.
        "Ns": [1] * 6,
        # Gauss-Hermite nodes per dimension; None for MC.
        "nodes": [None] * 6,
    },
    # Optimizer, control-variate, and stopping settings.
    "calibration": {
        # Maximum number of optimizer steps.
        "n_iter": 10_000,
        # Initial AdamW learning rate.
        "lr": 1e-4,
        # Highest stock-price power used as an MC control variate.
        "poly_cv_degree": 2,
        # Non-improving monitoring checks allowed before early stopping.
        "patience": 30,
        # Minimum monitored improvement needed to reset patience.
        "tol_error": 1e-7,
        # Optimizer steps before activating MC control variates.
        "warmup_without_cv": 0,
        # Best monitored loss must be below this to stop early.
        "limit_error_above": 1e-6,
        # Steps between fresh training MC draws; None keeps draws fixed.
        "simulate_every": 50,
        # AdamW decoupled weight-decay coefficient.
        "reg_lambda": 0,
        # Weight of the penalty toward the starting chaos coefficients.
        "anchor_lambda": 0,
        # Steps between fixed validation checks; None monitors training loss.
        "check_validation_set_MC": None,
        # Steps between price-loss and speed reports; None suppresses reports.
        "print_loss_every_it": 1,
        # Whether model.calibrate displays fit plots.
        "print_plots": False,
        # Learning-rate scheduler settings for stalled monitoring loss.
        "plateau_kwargs": {
            # Whether lower or higher monitored values are better.
            "mode": "min",
            # Multiplier applied to the learning rate on a plateau.
            "factor": 0.95,
            # Non-improving checks before reducing the learning rate.
            "patience": 15,
            # Minimum change counted as scheduler improvement.
            "threshold": 1e-7,
            # Interpret threshold as absolute or relative.
            "threshold_mode": "abs",
            # Checks to wait after a learning-rate reduction.
            "cooldown": 0,
            # Lowest learning rate allowed by the scheduler.
            "min_lr": 1e-6,
            # Smallest learning-rate change worth applying.
            "eps": 1e-8,
        },
    },
}
OUTPUT_DIR = STUDY_DIR / "Statistics calibration" / "single"

for split in ("calibr", "test"):
    pricing = CONFIG[f"pricing_{split}"]
    expected = len(data[f"maturities_{split}"])
    if any(len(pricing[key]) != expected for key in ("method", "N", "Ns", "nodes")):
        raise ValueError(f"Pricing settings for {split} must have {expected} entries.")

print(f"Device: {DEVICE} | Seed: {CONFIG['base_seed']} | Model: chaos {CONFIG['basis']}")
print(f"Initialization: {CONFIG['initialization']['type']} | Results: {OUTPUT_DIR}")
maturity_table = pd.DataFrame([
    {"set": label, "maturity": t, "pricing": method, "MC paths": count * repeats}
    for label, split in (("Calibration", "calibr"), ("Test", "test"))
    for t, method, count, repeats in zip(
        data[f"maturities_{split}"], CONFIG[f"pricing_{split}"]["method"],
        CONFIG[f"pricing_{split}"]["N"], CONFIG[f"pricing_{split}"]["Ns"],
    )
]).sort_values("maturity", ignore_index=True)
display(maturity_table)


## 3. Calibration routines

These helpers define the workflow and do not start calibration.


### Reproducibility, saving, and error metrics


In [ ]:
def seed_everything(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def json_safe(value):
    """Detach nested tensors and use JSON null for nonfinite diagnostics."""
    if torch.is_tensor(value):
        return json_safe(value.detach().cpu().tolist())
    if isinstance(value, np.ndarray):
        return json_safe(value.tolist())
    if isinstance(value, dict):
        return {str(k): json_safe(v) for k, v in value.items()}
    if isinstance(value, (list, tuple)):
        return [json_safe(v) for v in value]
    if isinstance(value, (float, np.floating)):
        return float(value) if np.isfinite(value) else None
    if isinstance(value, np.integer):
        return int(value)
    return value


def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(path.suffix + ".tmp")
    temporary.write_text(json.dumps(json_safe(payload), indent=2, allow_nan=False) + "\n")
    temporary.replace(path)


def surface_metrics(actual, expected):
    """Do not silently discard invalid IVs or mismatched surface entries."""
    if len(actual) != len(expected):
        raise ValueError("Different numbers of maturity rows.")
    errors = []
    invalid = 0
    for row, reference in zip(actual, expected):
        if len(row) != len(reference):
            raise ValueError("Different numbers of strikes.")
        for value, target in zip(row, reference):
            if value is None or not np.isfinite(value) or not np.isfinite(target):
                invalid += 1
            else:
                errors.append(abs(float(value) - float(target)))
    return {
        "mae": float(np.mean(errors)) if errors and not invalid else None,
        "invalid_quotes": invalid,
    }

### Model setup and held-out pricing


In [ ]:
def initialize_parameters_timed(model, **kwargs):
    """Time parameter initialization only, including completed GPU work."""
    if model.device.type == "cuda":
        torch.cuda.synchronize(model.device)
    init_started = time.perf_counter()
    initial = model.initialize_params(**kwargs)
    if model.device.type == "cuda":
        torch.cuda.synchronize(model.device)
    return initial, time.perf_counter() - init_started


def build_model(data, config):
    model = ModelStock(data["S0"], torch.device(config["device"]))
    horizon = max(data["maturities_calibr"] + data["maturities_test"])
    model.initialize_chaos(
        config["basis"], horizon, config["d"], config["P"], config["M"],
        base_jitter=config["base_jitter"],
    )
    model.set_up_for_Ts(data["maturities_calibr"], data["r"], data["q"])
    model.set_up_for_Ts(data["maturities_test"], data["r"], data["q"])
    return model


@torch.no_grad()
def test_surface(model, coefficients, data, pricing, calibration):
    prices = []
    for t, strikes, method, count, repetitions, nodes in zip(
        data["maturities_test"], data["strikes_test"], pricing["method"],
        pricing["N"], pricing["Ns"], pricing["nodes"],
    ):
        x = None
        if method == "MC":
            x = model._simulate_X_block([t], [method], [count], [repetitions])[0]
        price = model.compute_model_prices(
            coefficients, t, strikes, method, X=x, n=nodes,
            poly_cv_degree=calibration["poly_cv_degree"],
        )
        prices.append(price.detach().cpu().tolist())
    ivs = compute_ivs(
        prices, data["maturities_test"], data["strikes_test"],
        data["S0"], data["r"], data["q"],
    )
    return prices, ivs


## 4. Run one calibration

Fit Heston once for the warm start, calibrate the chaos coefficients, then price the held-out maturities. The calibration function reports its loss and iterations per second every 300 steps.


In [ ]:
progress = display(Markdown("Calibrating the model…"), display_id=True)
seed = CONFIG["base_seed"]
calibration_seed = seed + 3_000_000
test_seed = seed + 1_000_000
seed_everything(seed)
model = build_model(data, CONFIG)
pricing = CONFIG["pricing_calibr"]
init_config = CONFIG["initialization"]
if init_config["type"] == "calibrate_heston":
    progress.update(Markdown("Initializing Heston and projecting the terminal square-root factor…"))
    initial, init_seconds = initialize_parameters_timed(
        model,
        type="calibrate_heston",
        maturities=data["maturities_calibr"],
        strikes=data["strikes_calibr"],
        prices_market=data["prices_calibr"],
        r=data["r"], q=data["q"], seed=seed,
        **{key: value for key, value in init_config.items() if key != "type"},
    )
    print(f"Heston parameters: {model.heston_params}; "
          f"optimizer success: {getattr(model.heston_fit_result, 'success', 'skipped (heuristic)')}")
elif init_config["type"] == "random":
    initial, init_seconds = initialize_parameters_timed(model, type="random", scale=CONFIG["init_scale"])
else:
    raise ValueError(f"Unknown initialization type: {init_config['type']}")
fit = getattr(model, "heston_fit_result", None)
if fit is not None and not fit.success:
    raise RuntimeError(f"Heston fit did not converge: {fit.message}")
# Calibration updates the parameter in place; preserve the actual starting vector.
initial_coefficients = initial.detach().cpu().clone()
progress.update(Markdown("Calibrating the chaos model…"))

# Match the study: training has its own reproducible random stream.
seed_everything(calibration_seed)
d_a, prices_model, loss_ins, loss_oos, fit_seconds = model.calibrate(
    data["iv_calibr"], data["prices_calibr"], [data["maturities_calibr"]],
    data["strikes_calibr"], pricing["method"], pricing["N"], pricing["Ns"],
    pricing["nodes"], d_a=initial, **CONFIG["calibration"],
)
if not torch.isfinite(d_a).all():
    raise FloatingPointError("Nonfinite calibrated coefficients.")
iv_model = compute_ivs(
    prices_model, data["maturities_calibr"], data["strikes_calibr"],
    data["S0"], data["r"], data["q"],
)
progress.update(Markdown("Evaluating held-out maturities…"))
seed_everything(test_seed)
prices_model_test, iv_model_test = test_surface(
    model, d_a, data, CONFIG["pricing_test"], CONFIG["calibration"],
)
result = json_safe({
    "config": CONFIG, "seed": seed, "projection_seed": seed,
    "calibration_seed": calibration_seed, "test_seed": test_seed,
    "d_a": d_a,
    "initial_coefficients": initial_coefficients,
    "heston_init_params": getattr(model, "heston_params", None),
    "heston_initial_guess": getattr(model, "heston_initial_guess", None),
    "heston_fit": ({"success": bool(fit.success), "iterations": int(fit.nit),
                    "objective": float(fit.fun), "message": str(fit.message)}
                   if fit is not None else None),
    "calibration_iterations": model.last_calibration_iterations,
    "loss_ins": loss_ins, "loss_oos": loss_oos,
    "prices_model": prices_model, "prices_model_test": prices_model_test,
    "iv_model": iv_model, "iv_model_test": iv_model_test,
    "iv_calibration_metrics": surface_metrics(iv_model, data["iv_calibr"]),
    "iv_test_metrics": surface_metrics(iv_model_test, data["iv_test"]),
    "elapsed_seconds_init": init_seconds,
    "elapsed_seconds_chaos_calibration": fit_seconds,
    "elapsed_seconds_total": init_seconds + fit_seconds,
    "data_sha256": hashlib.sha256(DATA_PATH.read_bytes()).hexdigest(),
    "heston_source_sha256": hashlib.sha256(Path(heston_module.__file__).read_bytes()).hexdigest(),
    "model_source_sha256": hashlib.sha256(Path(pricing_module.__file__).read_bytes()).hexdigest(),
    "chaos_general_source_sha256": source_fingerprint(ROOT / "wiener_chaos/core/chaos.py"),
    "runtime": {"torch": torch.__version__, "numpy": np.__version__, "device": DEVICE},
})
# Preserve the numerical result even if a later plot fails.
save_json(OUTPUT_DIR / "result.json", result)
calibration_iv_mae = result["iv_calibration_metrics"]["mae"]
calibration_iv_text = (f"{calibration_iv_mae * 1e4:.2f} bp"
                       if calibration_iv_mae is not None else "unavailable (invalid IVs)")
test_iv_mae = result["iv_test_metrics"]["mae"]
test_iv_text = (f"{test_iv_mae * 1e4:.2f} bp"
                if test_iv_mae is not None else "unavailable (invalid IVs)")
progress.update(Markdown(
    f"Calibration complete in {result['elapsed_seconds_total']:.1f} seconds. "
    f"IV MAE: calibration {calibration_iv_text}; test {test_iv_text}."
))


## 5. Calibration quality and runtime

IV MAEs are in basis points of volatility (1 bp = 0.0001). Calibration-maturity prices use fresh Monte Carlo samples after fitting; test prices use held-out maturities. Invalid IV entries invalidate the corresponding MAE.

Separate tables below report calibration and test IV MAE across strikes at each
maturity, in bp, with quote counts and invalid-IV counts. These reuse the
already evaluated surfaces; they do not repeat calibration or pricing.


In [ ]:
fit_rows = []
for label, key in (
    ("Calibration", "iv_calibration_metrics"),
    ("Test", "iv_test_metrics"),
):
    metrics = result[key]
    fit_rows.append({
        "set": label,
        "iv_mae_bp": None if metrics["mae"] is None else 1e4 * metrics["mae"],
        "invalid_iv_quotes": metrics["invalid_quotes"],
    })
fit_summary = pd.DataFrame(fit_rows).set_index("set")
display(fit_summary.round(4))
print(f"Calibration time: {fit_seconds:.2f}s | Total with evaluation: {result['elapsed_seconds_total']:.2f}s")

# IV MAE across strikes at each maturity, using the prices already evaluated above.
maturity_rows = []
for label, split, fitted_key in (
    ("Calibration", "calibr", "iv_model"),
    ("Test", "test", "iv_model_test"),
):
    maturities = data[f"maturities_{split}"]
    fitted_surface = result[fitted_key]
    reference_surface = data[f"iv_{split}"]
    if not len(maturities) == len(fitted_surface) == len(reference_surface):
        raise ValueError(f"{label}: maturities and IV rows must be aligned.")
    for maturity, fitted, reference in zip(maturities, fitted_surface, reference_surface):
        metrics = surface_metrics([fitted], [reference])
        maturity_rows.append({
            "set": label,
            "maturity": maturity,
            "quotes": len(reference),
            "iv_mae_bp": None if metrics["mae"] is None else 1e4 * metrics["mae"],
            "invalid_iv_quotes": metrics["invalid_quotes"],
        })
fit_by_maturity = pd.DataFrame(maturity_rows).set_index(["set", "maturity"])
for label in ("Calibration", "Test"):
    display(Markdown(f"### {label} maturities — IV error in bp"))
    display(fit_by_maturity.xs(label, level="set").round(4))


In [ ]:
def plot_iv_surface(maturities, strikes, market_ivs, fitted_ivs, title):
    columns = 3
    rows = (len(maturities) + columns - 1) // columns
    fig, axes = plt.subplots(rows, columns, figsize=(13, 3.3 * rows), squeeze=False)
    for ax, maturity, strike_row, reference, fitted in zip(
        axes.flat, maturities, strikes, market_ivs, fitted_ivs
    ):
        ax.plot(strike_row, 100 * np.asarray(reference), "o", label="Heston target")
        ax.plot(strike_row, 100 * np.asarray(fitted, dtype=float), "-", label="Calibrated chaos")
        ax.set(title=f"T = {maturity:g}", xlabel="Strike", ylabel="Implied volatility (%)")
    for ax in list(axes.flat)[len(maturities):]:
        ax.set_visible(False)
    axes.flat[0].legend()
    fig.suptitle(title)
    fig.tight_layout()
    plt.show()
    return fig

figures = {}
for label, split, ivs in (("Calibration", "calibr", iv_model), ("Test", "test", iv_model_test)):
    figures["calibration" if split == "calibr" else "test"] = plot_iv_surface(
        data[f"maturities_{split}"], data[f"strikes_{split}"],
        data[f"iv_{split}"], ivs, f"{label} maturities: IV fit",
    )

## 6. Saved results

`Statistics calibration/single/result.json` records the configuration, fitted
coefficients, prices, implied volatilities, seeds, timings, and provenance.
`coefficients/` contains `initial_coefficients.pt` and `calibrated_coefficients.pt`;
`tables/` contains `fit_summary.csv` and `fit_by_maturity.csv`. Comparison figures
are saved in `figures/` as PNG and PDF files.


In [ ]:
save_artifacts(
    OUTPUT_DIR,
    coefficients={"initial_coefficients": result["initial_coefficients"],
                  "calibrated_coefficients": result["d_a"]},
    tables={"fit_summary": fit_summary, "fit_by_maturity": fit_by_maturity},
    figures=figures,
)
print(f"Single-calibration results saved in: {OUTPUT_DIR}")
